# Identify context-specific cofactors in different regions.

The ``predict_regulator_context_cofactors`` command trains a classifier to distinguish two sets of genomic regions, identifies regulatory factors that contribute most to the classification, and prioritizes context-specific cofactors for dual-functional target factors across the two contexts.

**Note**: The remaining examples will only show the direct command usage. 

If you need to use Singularity container, please refer to the [`singularity_use.ipynb`](singularity_use.ipynb) tutorial for detailed instructions on using `singularity exec` with `chrombert-tools`.

## Example:
EZH2, the catalytic subunit of Polycomb Repressive Complex 2 (PRC2), operates in two distinct modes: 
a classical H3K27me3-dependent repressive role and a non-classical H3K27me3-independent role.

We fine-tuned ChromBERT to classify EZH2 ChIP-seq peaks in human embryonic stem cells 
into classical and non-classical categories.

Using this fine-tuned model, we identify regulators that most strongly distinguish the two region sets and visualize context-specific EZH2 cofactor subnetworks.


In [7]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]='0'

In [ ]:
!chrombert-tools -h

Usage: chrombert-tools [OPTIONS] COMMAND [ARGS]...

  Type -h or --help after any subcommand for more information.

Options:
  -v, --verbose  Verbose logging
  -d, --debug    Post mortem debugging
  -V, --version  Show the version and exit.
  -h, --help     Show this message and exit.

Commands:
  embed_cell_cistrome             Extract cell-specific cistrome...
  embed_cell_gene                 Extract cell-specific gene embeddings
  embed_cell_region               Extract cell-specific region embeddings
  embed_cell_regulator            Extract cell-specific regulator...
  embed_cistrome                  Extract general cistrome embeddings on...
  embed_gene                      Extract general gene embeddings
  embed_region                    Extract general region embeddings
  embed_regulator                 Extract general regulator embeddings on...
  find_context_specific_cofactor  Find context-specific cofactors in...
  find_driver_in_transition       Find driver factors in cell

In [3]:
!chrombert-tools predict_regulator_context_cofactors -h

Usage: chrombert-tools predict_regulator_context_cofactors 
           [OPTIONS]

  Identify context-specific cofactors across two or more region-function
  classes for user-specified dual-functional regulators.

  Typical usage

  Provide two or more functional region classes with --function-bed to define
  the region groups to be compared.

  Provide --dual-regulator to specify the dual-functional regulator of
  interest. The workflow then compares cofactor patterns across the defined
  region classes for that regulator.

  Output

  For every pair of classes, the workflow writes results to:
  {odir}/results/<label_pair_subdir>/

Options:
  --function-bed TEXT             BED file(s) defining one functional region
                                  class. Repeat this option to provide
                                  multiple classes. Use ';' to combine
                                  multiple BED files into the same class. If
                                  only one class is pro

## Run

In [4]:
!mkdir -p ./tmp

In [9]:
# takes approximately 20-40 minutes to run
!chrombert-tools predict_regulator_context_cofactors \
    --function-bed "../data/hESC_GSM1003524_EZH2.bed;../data/hESC_GSM1498900_H3K27me3.bed" \
    --function-bed "../data/hESC_GSM1003524_EZH2.bed" \
    --function-name "EZH2-H3K27me3" \
    --function-name "EZH2" \
    --dual-regulator "EZH2" \
    --ignore-regulator "H3K27me3;H3K27me3/H3K4me3" \
    --odir "./output_find_context_specific_cofactor" \
    --genome "hg38" \
    --resolution "1kb"  2> "./tmp/predict_regulator_context_cofactors.log" # redirect stderr to log file

Note: All regulator names were converted to lowercase for matching.
Regulator count summary - requested: 2, matched in ChromBERT: 2, not found: 0, not found regulator: []
ChromBERT regulators: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt
Note: All regulator names were converted to lowercase for matching.
Regulator count summary - requested: 1, matched in ChromBERT: 1, not found: 0, not found regulator: []
ChromBERT regulators: /mnt/Storage/home/chenqianqian/.cache/chrombert/data/config/hg38_6k_regulators_list.txt
Step 1/3: Preparing labeled region dataset...
[EZH2-H3K27me3 | hESC_GSM1003524_EZH2.bed] Region summary - total: 11896, overlapping with ChromBERT: 12101 (one region may overlap multiple ChromBERT regions, we keep overlaps with ≥50% coverage of either the ChromBERT bin or the input region), non-overlapping: 325
[EZH2-H3K27me3 | hESC_GSM1498900_H3K27me3.bed] Region summary - total: 8875, overlapping with ChromBERT: 14135 (one region ma

In [17]:
# regulator_cosine_similarity_on_function1_regions.csv: cosine similarity of regulator-regulator pairs on function1 regions:
#   - node1: regulator name
#   - node2: regulator name
#   - similarity: cosine similarity of regulator embeddings between function1 regions


# regulator_cosine_similarity_on_function2_regions.csv: cosine similarity of regulator-regulator pairs on function2 regions:
#   - node1: regulator name
#   - node2: regulator name
#   - similarity: cosine similarity of regulator embeddings between function2 regions
import pandas as pd
reg_cos_sim_func1 = pd.read_csv("./output_find_context_specific_cofactor/results/0_1_EZH2-H3K27me3_vs_EZH2/func1/regulator_cosine_similarity.tsv",sep="\t",index_col=0)
reg_cos_sim_func2 = pd.read_csv("./output_find_context_specific_cofactor/results/0_1_EZH2-H3K27me3_vs_EZH2/func2/regulator_cosine_similarity.tsv",sep="\t",index_col=0)



In [18]:
reg_cos_sim_func2

,5hmc,adnp,aebp2,aff1,aff4,ago1,ago2,ahr,ahrr,alkbh3,...,zscan20,zscan22,zscan23,zscan29,zscan31,zscan5a,zta,zxdb,zxdc,zzz3
5hmc,1.000000,0.263275,0.312532,0.201742,0.192320,0.279180,0.241895,0.216915,0.186114,0.227855,...,0.383006,0.135186,0.437668,0.285964,0.172695,0.350433,0.288472,0.293085,0.213696,0.296846
adnp,0.263275,1.000000,0.656210,0.365337,0.493897,0.160843,0.232765,0.294296,0.314429,0.291633,...,0.393143,0.353003,0.448869,0.544043,0.380984,0.447373,0.335509,0.425058,0.357726,0.430030
aebp2,0.312532,0.656210,1.000000,0.280973,0.423670,0.248505,0.309748,0.337397,0.399152,0.321023,...,0.413266,0.235316,0.409336,0.375150,0.309375,0.331573,0.342340,0.296498,0.371933,0.367972
aff1,0.201742,0.365337,0.280973,1.000000,0.666506,0.286647,0.303003,0.340900,0.345766,0.246487,...,0.348551,0.305927,0.284400,0.351595,0.395420,0.321056,0.273604,0.273012,0.240874,0.334227
aff4,0.192320,0.493897,0.423670,0.666506,1.000000,0.270260,0.341720,0.328680,0.373705,0.330691,...,0.371609,0.457508,0.395476,0.405121,0.454660,0.382793,0.412013,0.357532,0.387004,0.416138
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zscan5a,0.350433,0.447373,0.331573,0.321056,0.382793,0.197964,0.222432,0.256884,0.258526,0.240125,...,0.800244,0.448363,0.864130,0.472762,0.524527,1.000000,0.263888,0.631232,0.378192,0.234377
zta,0.288472,0.335509,0.342340,0.273604,0.412013,0.152200,0.206198,0.247921,0.304893,0.282786,...,0.301469,0.245819,0.289613,0.258759,0.282314,0.263888,1.000000,0.166201,0.248931,0.462602
zxdb,0.293085,0.425058,0.296498,0.273012,0.357532,0.254524,0.258319,0.234817,0.229969,0.192176,...,0.605518,0.551410,0.622243,0.461396,0.460363,0.631232,0.166201,1.000000,0.412470,0.193181
zxdc,0.213696,0.357726,0.371933,0.240874,0.387004,0.256106,0.270268,0.615022,0.561938,0.310942,...,0.365331,0.326439,0.403753,0.361857,0.304385,0.378192,0.248931,0.412470,1.000000,0.234095


In [19]:
# Infer dual-functional regulator subnetwork (If --dual-regulator was provided, saved in {odir}/results/dual_regulator_subnetwork.pdf)
import numpy as np
thre_func1 = np.percentile(reg_cos_sim_func1.values.flatten(), 95)
thre_func2 = np.percentile(reg_cos_sim_func2.values.flatten(), 95)

assert (reg_cos_sim_func1.index == reg_cos_sim_func2.index).all()
df_cos_reg = pd.DataFrame(
                index=reg_cos_sim_func1.index,
                data={
                    "function1": reg_cos_sim_func1.loc['ezh2', :], # ezh2 (dual-functional regulator)
                    "function2": reg_cos_sim_func2.loc['ezh2', :], # ezh2 (dual-functional regulator)
                },
            )
df_cos_reg["diff"] = df_cos_reg["function1"] - df_cos_reg["function2"]
df_candidate = df_cos_reg[df_cos_reg["diff"].abs() > 0.1]
topN_pos = df_candidate.query("function1 > @thre_func1").index.values
topN_neg = df_candidate.query("function2 > @thre_func2").index.values
top_pairs = np.union1d(topN_pos, topN_neg)



In [20]:
df_candidate

,function1,function2,diff
arnt2,0.257709,0.368453,-0.110744
arntl,0.343475,0.462813,-0.119337
atf3,0.430633,0.533162,-0.102530
barhl1,0.295013,0.425979,-0.130966
batf,0.340393,0.458829,-0.118436
...,...,...,...
znf433,0.220247,0.330745,-0.110498
znf571,0.252793,0.367852,-0.115058
znf706,0.262874,0.376015,-0.113141
znf860,0.201699,0.338688,-0.136989


In [21]:
thre_func1, thre_func2

(0.6205606986763784, 0.5701609889810152)

In [22]:
# function1_sunbetwork
df_candidate.loc[topN_pos]

,function1,function2,diff
h2ak119ub,0.724501,0.621556,0.102945
pcgf1,0.653966,0.550944,0.103022


In [23]:
# function2_sunbetwork
df_candidate.loc[topN_neg]

,function1,function2,diff
brca1,0.482059,0.608059,-0.125999
ep300,0.496610,0.612843,-0.116234
foxm1,0.469633,0.625345,-0.155713
h2ak119ub,0.724501,0.621556,0.102945
hinfp,0.491919,0.600521,-0.108602
junb,0.445172,0.579780,-0.134608
kat5,0.469790,0.572971,-0.103180
med1,0.467571,0.589410,-0.121839
rela,0.474658,0.591733,-0.117075
smad4,0.447686,0.578598,-0.130911
